# Comparison between GradientBoostingClassifier and XGBClassifier as two variants of the CID_SID ML model 

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score
from statsmodels.stats.contingency_tables import mcnemar

# Set random seed for reproducibility
np.random.seed(42)


# All data preparation happens in the main block below.
def train_and_predict(X_train, X_test, y_train, y_test):
    """Trains two models and generates predictions on the shared test set."""
    print("\n--- 2. Model Training and Prediction ---")
    
    # Model A: Gradient Boosting Classifier
    model_A = GradientBoostingClassifier(n_estimators=100, random_state=42)
    model_A.fit(X_train, y_train)
    y_pred_A = model_A.predict(X_test)
    
    # Model B: XGBoost Classifier
    model_B = XGBClassifier(eval_metric='logloss', random_state=42)
    # The y_train is passed as a 1D array (ravelled)
    model_B.fit(X_train, y_train)
    y_pred_B = model_B.predict(X_test)
    
    acc_A = accuracy_score(y_test, y_pred_A)
    acc_B = accuracy_score(y_test, y_pred_B)

    print(f"Model A (GradientBoostingClassifier) Accuracy: {acc_A:.4f}")
    print(f"Model B (XGBoost) Accuracy: {acc_B:.4f}")
    
    return y_pred_A, y_pred_B

def perform_mcnemar_test(y_true, y_pred_A, y_pred_B, alpha=0.05):
    """
    Performs McNemar's Test to statistically compare two models.
    """
    print("\n--- 3. Statistical Comparison (McNemar's Test) ---")
    
    # n_01: Count where Model A was CORRECT and Model B was WRONG
    n_01 = np.sum((y_pred_A == y_true) & (y_pred_B != y_true))
    
    # n_10: Count where Model A was WRONG and Model B was CORRECT
    n_10 = np.sum((y_pred_A != y_true) & (y_pred_B == y_true))
    
    # Calculate the full 2x2 table for statsmodels
    n_00 = np.sum((y_pred_A == y_true) & (y_pred_B == y_true))
    n_11 = np.sum((y_pred_A != y_true) & (y_pred_B != y_true))
    
    contingency_table = np.array([[n_00, n_01],
                                  [n_10, n_11]])
    
    print(f"Disagreement Counts (A Correct/B Wrong): n_01 = {n_01}")
    print(f"Disagreement Counts (A Wrong/B Correct): n_10 = {n_10}")
    
    # Perform the test
    result = mcnemar(contingency_table, exact=False)
    p_value = result.pvalue

    # 4. Interpret the Results
    print(f"\nSignificance Level (alpha): {alpha}")
    print(f"McNemar's Test P-value: {p_value:.4f}")
    
    if p_value < alpha:
        conclusion = "The difference in performance is **STATISTICALLY SIGNIFICANT**."
        if n_01 > n_10:
            conclusion += " (Model A is significantly better than Model B.)"
        else:
            conclusion += " (Model B is significantly better than Model A.)"
    else:
        conclusion = "The difference in performance is **NOT STATISTICALLY SIGNIFICANT**."
        conclusion += " (The observed difference is likely due to random chance.)"

    print("\nConclusion:")
    print(conclusion)

if __name__ == '__main__':
    print("--- 1. Data Preparation and Split ---")
    
    df = pd.read_csv('data_IUPACs_II.csv', index_col=[0])
    
    # 1. Shuffle the entire DataFrame first
    df = df.sample(frac=1).reset_index(drop=True)
    
    # 2. Define features (X) and target (y)
    X = df.drop(columns='target', axis=1)
    # Target is converted to a 1D array for scikit-learn compatibility
    y = df['target'].values.ravel() 

    # 3. Split the data into training and testing sets
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.3, random_state=42, stratify=y
    )
    
    # y_true is just the true labels from the test set
    y_true = y_test 
    
    print(f"Total Samples: {len(df)}")
    print(f"Training Samples: {len(X_train)}")
    print(f"Testing Samples (Shared Test Set): {len(X_test)}")
    
    # Train and Get Predictions
    y_pred_A, y_pred_B = train_and_predict(X_train, X_test, y_train, y_test) 
    
    # Perform Statistical Comparison
    perform_mcnemar_test(y_true, y_pred_A, y_pred_B)


--- 1. Data Preparation and Split ---
Total Samples: 101860
Training Samples: 71302
Testing Samples (Shared Test Set): 30558

--- 2. Model Training and Prediction ---
Model A (GradientBoostingClassifier) Accuracy: 0.6547
Model B (XGBoost) Accuracy: 0.7280

--- 3. Statistical Comparison (McNemar's Test) ---
Disagreement Counts (A Correct/B Wrong): n_01 = 1784
Disagreement Counts (A Wrong/B Correct): n_10 = 4023

Significance Level (alpha): 0.05
McNemar's Test P-value: 0.0000

Conclusion:
The difference in performance is **STATISTICALLY SIGNIFICANT**. (Model B is significantly better than Model A.)
